# 09 - Plugin System and Hook Lifecycle

> **When to use**: When you need to extend sqlseed's functionality (e.g., auto-add timestamps, data masking, custom generators).
>
> **Core concept**: pluggy plugin framework, 12 declared hook contracts, including active batch transformation.

## Applicable Scenarios

- Auto-add fields in the generated batch (e.g., `created_at`) → `sqlseed_transform_batch`
- Batch data transformation (e.g., uppercase all) → `transform_batch` Hook
- Custom data generators → `register_providers` Hook
- Run logic before/after writing → `before_insert` / `after_insert` Hook

## What You Will Learn

- 12 declared hook contracts
- Custom Provider development
- PluginMediator bridging mechanism
- entry-point packaging flow

See architecture.zh-CN.md §8

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 01 |
| **→ 09** | **Plugin System and Hook Lifecycle** | **Plugins** | **01** |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [ ]:
from __future__ import annotations

# Run from examples/notebooks. Install from the repository root in one resolution:
# python -m pip install -e ".[dev,all]" -e "./plugins/sqlseed-cli" \
#   -e "./plugins/sqlseed-ai[dev,mcp]" -e "./plugins/mcp-server-sqlseed" -e "./plugins/sqlseed-web[dev]"
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

import sqlseed
from sqlseed import connect

sys.path.insert(0, str(Path("..").resolve()))  # build_demo_db only
from build_demo_db import build

# Keep this object alive across cells. No existing database is opened or rebuilt.
_demo_directory = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-")
demo_root = Path(_demo_directory.name)
os.environ["SQLSEED_CACHE_DIR"] = str(demo_root / "cache")
db_path = build(demo_root / "demo.db")


def require(condition, message):
    """Stop the tutorial if an expected outcome did not occur."""
    if not condition:
        raise RuntimeError(message)


def check_generation(result, count):
    """Check errors and generated row count before showing success."""
    require(not result.errors and result.count == count, f"Generation failed: {result.errors}; count={result.count}")


def read_rows(database, sql):
    """Read actual persisted values using a fixed tutorial query."""
    # The queries below are fixed tutorial SQL, never external identifiers.
    with closing(sqlite3.connect(database)) as connection:
        return connection.execute(sql).fetchall()


with connect(str(db_path), provider="faker") as orch:
    for table, count in (("organizations", 5), ("members", 20), ("projects", 10), ("tags", 8)):
        check_generation(orch.fill_table(table, count=count, seed=42, skip_ai=True), count)

print(f"sqlseed {sqlseed.__version__} | Temporary database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Plugin Hook Specs | `src/sqlseed/plugins/hookspecs.py` | `SqlseedHookSpec` |

> Corresponding architecture diagram: [§8 Plugin Hook Lifecycle](../../docs/architecture.zh-CN.md#8-插件-hook-生命周期)

## 1. Add Timestamps with the Active Batch Hook

The current orchestrator calls `sqlseed_transform_batch` before insertion. The example below adds a timestamp to each row in that batch and checks the persisted values. The separately declared `sqlseed_transform_row` hook is not currently invoked by the fill path.


## 2. Custom Provider Development

A custom Provider must implement the `DataProvider` Protocol and inherit from `GeneratorDispatchMixin`:

In [ ]:
import random
from typing import ClassVar

from sqlseed.generators._dispatch import GeneratorDispatchMixin
from sqlseed.generators._protocol import DataProvider


class ChineseNameProvider(GeneratorDispatchMixin, DataProvider):
    """Provide seeded Chinese names as a minimal custom provider."""

    name = "chinese_name"
    GENERATOR_MAP: ClassVar[dict] = {}

    def __init__(self):
        """Create the name lists and dedicated random generator."""
        self._locale = "zh"
        self._random = random.Random()
        self._last_names = ["张", "王", "李", "赵", "刘"]
        self._first_names = ["伟", "芳", "秀英", "敏", "静"]

    def set_locale(self, locale):
        """Store the selected locale for this demonstration provider."""
        self._locale = locale

    def set_seed(self, seed):
        """Reset the provider random state for reproducibility."""
        self._random.seed(seed)

    def generate(self, type_name, **params):
        """Generate a name or reject an unsupported generator."""
        if type_name != "name":
            raise ValueError(f"Unsupported generator: {type_name}")
        return self._random.choice(self._last_names) + self._random.choice(self._first_names)


provider = ChineseNameProvider()
provider.set_seed(42)
first = [provider.generate("name") for _ in range(3)]
provider.set_seed(42)
require([provider.generate("name") for _ in range(3)] == first, "Provider did not honor the seed")
print(first)

## 3. 12 Hook Lifecycle

sqlseed declares 12 hook contracts. This table groups their intended phases; declaration alone does not prove an orchestrator call site. The `sqlseed_transform_row` contract is not called by current fill orchestration, so the examples below use `sqlseed_transform_batch` for real writes.

| Phase | Hook | Description |
|---|---|---|
| Register | sqlseed_register_providers | Register custom Provider |
| Register | sqlseed_register_column_mappers | Register custom column mapping rules |
| AI Analysis | sqlseed_apply_ai_suggestions | Apply AI suggestions to column mapping (firstresult) |
| AI Analysis | sqlseed_ai_analyze_table | AI analyzes table structure (firstresult) |
| AI Analysis | sqlseed_pre_generate_templates | Pre-generate template values (firstresult) |
| Generate | sqlseed_before_generate | Pre-generation callback |
| Generate | sqlseed_after_generate | Post-generation callback |
| Generate | sqlseed_transform_row | Per-row transform contract (not invoked by current fill orchestration) |
| Generate | sqlseed_transform_batch | Batch transform (last non-None returned result) |
| Write | sqlseed_before_insert | Pre-insert callback |
| Write | sqlseed_after_insert | Post-insert callback |
| Shared Pool | sqlseed_shared_pool_loaded | Shared pool load complete |

In [ ]:
from sqlseed.plugins.hookspecs import SqlseedHookSpec

hookspec_names = [
    ("sqlseed_register_providers", "Register", "Register custom Provider"),
    ("sqlseed_register_column_mappers", "Register", "Register custom column mapping rules"),
    ("sqlseed_apply_ai_suggestions", "AI Analysis", "Apply AI suggestions to mapping (firstresult)"),
    ("sqlseed_ai_analyze_table", "AI Analysis", "AI analyzes table structure (firstresult)"),
    ("sqlseed_pre_generate_templates", "AI Analysis", "Pre-generate template values (firstresult)"),
    ("sqlseed_before_generate", "Generate", "Pre-generation callback"),
    ("sqlseed_after_generate", "Generate", "Post-generation callback"),
    ("sqlseed_transform_row", "Generate", "Per-row transform contract (not invoked by current fill orchestration)"),
    ("sqlseed_transform_batch", "Generate", "Batch transform (last non-None returned result)"),
    ("sqlseed_before_insert", "Write", "Pre-insert callback"),
    ("sqlseed_after_insert", "Write", "Post-insert callback"),
    ("sqlseed_shared_pool_loaded", "Shared Pool", "Shared pool load complete"),
]

actual_hooks = {name for name in vars(SqlseedHookSpec) if name.startswith("sqlseed_")}
require(actual_hooks == {item[0] for item in hookspec_names}, "Hook list differs from the installed specification")
print(f"sqlseed {len(actual_hooks)} Hook lifecycle:\n")
for i, (name, phase, desc) in enumerate(hookspec_names, 1):
    print(f"  {i:2d}. [{phase}] {name}")
    print(f"      {desc}")

## 4. In Practice: Timestamp a Batch

Register a batch plugin and generate a dedicated table. This leaves the existing organizations and their dependent rows unchanged.


In [ ]:
from datetime import datetime, timezone

from sqlseed.plugins.hookspecs import hookimpl


class TimestampPlugin:
    """Add a timestamp to every row in the dedicated demonstration table."""

    @hookimpl
    def sqlseed_transform_batch(self, table_name, batch):
        """Transform the batch in the orchestrator's active plugin call path."""
        if table_name == "hook_rows":
            for row in batch:
                row["created_at"] = datetime.now(timezone.utc).isoformat()
        return batch


with closing(sqlite3.connect(db_path)) as connection:
    connection.execute("CREATE TABLE IF NOT EXISTS hook_rows(id INTEGER PRIMARY KEY, name TEXT, created_at TEXT)")
    connection.commit()

with connect(str(db_path), provider="faker") as orch:
    # Internal integration surface; installed packages use the entry point below.
    orch._ext.plugins.register(TimestampPlugin())
    result = orch.fill_table("hook_rows", count=5, clear_before=True, seed=42, skip_ai=True)
check_generation(result, 5)
rows = read_rows(db_path, "SELECT name, created_at FROM hook_rows ORDER BY id")
require(len(rows) == 5, "Missing transformed rows")
require(all(datetime.fromisoformat(row[1]).tzinfo is not None for row in rows), "UTC timestamp hook did not run")
print(rows)

## 5. transform_batch Hook

`transform_batch` transforms the entire batch of data, suitable for batch computation or filtering.

In [ ]:
class UppercasePlugin:
    """Uppercase names in the demonstration batch."""

    @hookimpl
    def sqlseed_transform_batch(self, table_name, batch):
        """Change names while retaining the other generated columns."""
        if table_name == "hook_rows":
            for row in batch:
                row["name"] = row["name"].upper()
        return batch


with connect(str(db_path), provider="faker") as orch:
    orch._ext.plugins.register(UppercasePlugin())
    result = orch.fill_table(
        "hook_rows",
        count=3,
        clear_before=True,
        seed=42,
        skip_ai=True,
        columns={"name": {"type": "choice", "choices": ["Ada", "Lin"]}},
    )
check_generation(result, 3)
rows = read_rows(db_path, "SELECT name FROM hook_rows ORDER BY id")
require(
    len(rows) == 3 and all(row[0] in {"ADA", "LIN"} for row in rows), "Batch transform did not persist uppercase names"
)
print(rows)

## 6. Package as a Standalone Plugin

Via the `entry-point` mechanism, sqlseed auto-discovers and loads installed plugin packages.

In [ ]:
# Minimal pyproject.toml for a sqlseed plugin
print('''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "sqlseed-my-plugin"
version = "0.1.0"
dependencies = ["sqlseed>=0.2.4,<0.3"]

[project.entry-points.sqlseed]
my_plugin = "my_plugin.plugin:plugin"

[tool.hatch.build.targets.wheel]
packages = ["my_plugin"]''')

print("\nPlugin module (my_plugin/plugin.py):")
print("""from sqlseed.plugins.hookspecs import hookimpl

class MyPlugin:
    @hookimpl
    def sqlseed_transform_batch(self, table_name, batch):
        for row in batch:
            if "name" in row:
                row["name"] = row["name"].upper()
        return batch

plugin = MyPlugin()""")

## Summary

| Hook | Phase | Purpose |
|------|------|------|
| `sqlseed_register_providers` | Register | Custom Provider |
| `sqlseed_register_column_mappers` | Register | Custom column mapping |
| `sqlseed_before_generate` | Pre-generate | Preparation |
| `sqlseed_transform_row` | Declared contract | Not invoked by current fill orchestration |
| `sqlseed_transform_batch` | During generate | Batch transform |
| `sqlseed_after_generate` | Post-generate | Cleanup |
| `sqlseed_before_insert` | Pre-insert | Preprocessing |
| `sqlseed_after_insert` | Post-insert | Record stats |

**Next**: [10-cli-reference.ipynb](10-cli-reference.ipynb) — CLI Reference Manual

In [ ]:
require(len(read_rows(db_path, "SELECT member_id FROM members")) == 20, "The examples changed unrelated members")
print("Tutorial operations completed with the original 20 demo members preserved.")